# P8｜Eligibility-Masked Objective

**状态：Design / Not Ready；8/20 core。** 目的：只在语义兼容且有合法观测的 lanes/rows 上加入 compatible supervision，同时保留各数据集 native objective。

## 唯一变量、匹配对照与四数据集边界

matched comparator 固定为 P1。P8 与 P1 的 frozen AST encoder、shared projector 768→256、dataset-native heads、source-proportional sampler、trainable scope、split、update budget、seed=20260728 与 selection 必须完全匹配；唯一变量是 eligibility-aware auxiliary objective。missing、unknown、not-annotated、empty 或 gap 一律不构造 negative。

- 只有 ICBHI cycle flat4 与 SPRSound event binary/raw7 中明确定义且兼容的 shared attributes/labels 可进入 eligible auxiliary supervision。
- HF time annotation 与 KAUH recording raw9 只保留 native objective，不进入 shared projector auxiliary evidence，也不得计为 shared-label evidence；KAUH shared mapping/diagnosis HOLD。
- 输入为 P1 representations/native targets + ICBHI/SPRSound eligibility/observation masks；输出为四数据集 native logits、仅两条 eligible lanes 的 auxiliary logits/loss 与 per-lane counts。
- 必须报告 native retention 和 worst-task guardrail；不得用 pooled score 掩盖单任务退化。

In [ ]:
from pathlib import Path
import os

PIPELINE = {
    "id": "P8",
    "comparator": "P1",
    "only_change": "eligible_auxiliary_objective_on_ICBHI_and_SPRSound",
    "seed": 20260728,
    "split_policy": "reuse_P1_immutable_receipts",
    "eligible_auxiliary_lanes": ["ICBHI", "SPRSound"],
    "native_only_lanes": ["HF", "KAUH"],
    "matched_scope": "P1_encoder_projector_heads_sampler_scope_budget_seed_selection",
    "negative_policy": "observed_negative_only",
    "native_retention": True,
    "contract_modules": ["baseline.multidataset_pipeline.contracts", "baseline.multidataset_pipeline.eligibility"],
    "engineering_test": "tests/test_multidataset_pipeline.py::EligibilityObjectiveTest",
    "worst_task_guardrail": None,
    "update_budget": None,
    "selection": None,
    "output_dir": "result/reproduce/P8_eligibility_masked_objective",
    "receipt_path": "result/reproduce/P8_eligibility_masked_objective/P8_receipt.json",
}
PROJECT_ROOT = Path(os.environ.get("ACOUSTIC_PROJECT_ROOT", Path.cwd())).resolve()
APPROVAL_RECEIPT = os.environ.get("P8_APPROVAL_RECEIPT")


## 科学 gate 与运行前审批

冻结 P1 parity hash、ICBHI/SPRSound compatibility table、row eligibility、loss normalization、native-retention tolerance、worst-task go/no-go、budget、metrics 与 selection；验证 HF/KAUH auxiliary calls 为零、masked rows 零梯度、eligible counts 非零且可复算、missing/gap 不进入负例。outer/test 必须在 selection 后 terminal scoring。审批 receipt 缺失时 fail closed。

In [ ]:
required_keys = ["comparator", "worst_task_guardrail", "update_budget", "selection"]
if any(PIPELINE[key] in (None, "") for key in required_keys) or not APPROVAL_RECEIPT:
    raise RuntimeError("P8 fail closed: comparator, guardrail, budget, selection, and approval must be frozen")
approval_path = PROJECT_ROOT / APPROVAL_RECEIPT
if not approval_path.is_file():
    raise FileNotFoundError(approval_path)
DRY_RUN_PLAN = {"pipeline": PIPELINE, "execute": False}


## 输出、receipt 与结果表

receipt 含 eligibility table/hash、per-dataset/lane eligible/masked/observed-negative counts、native/auxiliary loss、native metrics、worst-task delta、seed/updates、selection 与 verifier warnings。

| Comparator | Native retention | Worst-task guardrail | Result |
|---|---:|---:|---:|
| P1 | Not run | Not run | Not run |

**Test Result = Not run。Decision = Not made。Claim boundary：shared-label evidence 仅来自 ICBHI+SPRSound 的 compatible eligible rows；HF/KAUH 仅 native，missing/unknown 不是 negative。**